In [ ]:
# Step 1: 安装依赖（如果需要）
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "openai", "python-dotenv", "-q"])
print("依赖安装完成！")

In [ ]:
# Step 2: 设置项目路径（动态识别）
import sys
from pathlib import Path

cwd = Path.cwd()

# 如果在 notebooks 目录，项目根目录是上一级
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    # 向上查找 pyproject.toml
    project_root = cwd
    while not (project_root / "pyproject.toml").exists():
        project_root = project_root.parent

src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"源码路径: {src_path}")

# LLM 交互基础教程

本教程帮助你理解 LLM 交互的核心概念。

## 学习目标

1. 理解 LLM API 的本质
2. 掌握消息模型（Message, Role）
3. 完成第一次 LLM 调用
4. 理解多轮对话的原理
5. 理解抽象层的价值

## 1. 什么是 LLM API？

LLM（大语言模型）API 本质上是**消息的交换**：

```
你发送: [消息1, 消息2, 消息3, ...]
LLM 返回: AI 的回复
```

就这么简单！

In [ ]:
# 导入消息模型
from rogue_rabbit.contracts import Message, Role
from rogue_rabbit.adapters import GLMClient

print("导入成功！")

# 查看有哪些角色
print("\n可用的角色：")
for role in Role:
    print(f"  - {role.name}: {role.value}")

### 三种角色的含义

| 角色 | 含义 | 示例 |
|------|------|------|
| SYSTEM | 系统指令，定义 AI 行为 | "你是一个有帮助的助手" |
| USER | 用户输入，人类的问题 | "什么是机器学习？" |
| ASSISTANT | AI 回复，模型的输出 | "机器学习是..." |

In [ ]:
# 创建消息
user_message = Message(role=Role.USER, content="你好")

print(f"消息: {user_message}")
print(f"角色: {user_message.role.name}")
print(f"内容: {user_message.content}")

### 为什么用 dataclass？

```python
@dataclass(frozen=True)
class Message:
    role: Role
    content: str
```

- **简洁**: 自动生成 `__init__`, `__repr__` 等方法
- **不可变** (`frozen=True`): 消息一旦创建就不能修改，更安全
- **类型安全**: 明确的字段类型

## 2. 第一次 LLM 调用

现在让我们真正调用 LLM！

In [ ]:
# 创建客户端（自动从 .env 读取 ZHIPU_API_KEY）
client = GLMClient()

# 创建消息
messages = [
    Message(role=Role.USER, content="用中文说你好，最多5个字")
]

# 调用 LLM
print("正在发送请求...")
response = client.complete(messages)

print(f"用户: {messages[0].content}")
print(f"AI: {response}")

### 调用流程图

```
+-------------+
| 1.创建消息  | Message(role=USER, content="...")
+------+------+
       |
       v
+-------------+
| 2.创建客户端| GLMClient()
+------+------+
       |
       v
+-------------+
| 3.调用 LLM  | client.complete(messages)
+------+------+
       |
       v
+-------------+
| 4.获取回复  | response: str
+-------------+
```

## 3. 多轮对话

LLM 是**无状态**的，每次调用都需要发送完整的对话历史。

In [ ]:
# 多轮对话示例
messages = [
    Message(role=Role.USER, content="我叫小明"),
    Message(role=Role.ASSISTANT, content="你好小明，很高兴认识你！"),
    Message(role=Role.USER, content="我叫什么名字？"),
]

response = client.complete(messages)
print(f"AI: {response}")
print("\n注意：AI 记住了名字，因为我们发送了完整的对话历史！")

### 为什么需要发送完整历史？

```
LLM 不记住任何东西！

每次调用都是独立的：
- 调用 1: [用户: 我叫小明] -> AI: 你好小明
- 调用 2: [用户: 我叫小明, AI: 你好小明, 用户: 我叫什么] -> AI: 你叫小明
                ^
          必须带上历史！
```

## 4. 理解抽象层的价值

我们设计了两个层次：`contracts`（协议）和 `adapters`（实现）。

In [ ]:
# 场景：写一个通用的问答函数
from rogue_rabbit.contracts import LLMClient

def ask_question(client: LLMClient, question: str) -> str:
    """
    通用的问答函数
    参数 client 的类型是 LLMClient（协议）
    这意味着任何实现了 complete 方法的对象都可以传入
    """
    messages = [Message(role=Role.USER, content=question)]
    return client.complete(messages)

# 使用 GLM 客户端
print("GLM:", ask_question(client, "1+1=?")[:50])

In [ ]:
# 使用 Mock 客户端（不需要 API key）
from rogue_rabbit.contracts import MockLLMClient

mock_client = MockLLMClient(response="这是一个模拟响应")
print("Mock:", ask_question(mock_client, "1+1=?"))

### 分层的好处

```
+----------------------------------------+
|           业务代码                      |
|   ask_question(client, question)       |
+----------------+-----------------------+
                 | 只依赖协议
                 v
+----------------------------------------+
|         contracts (协议层)              |
|   LLMClient: complete(messages) -> str |
+----------------+-----------------------+
                 | 多种实现
        +--------+--------+
        v                 v
+-------------+   +-------------+
| GLM         |   | Mock        |
| Adapter     |   | Client      |
+-------------+   +-------------+
```

1. **解耦**: 业务代码不依赖具体实现
2. **可替换**: 可以随时切换 LLM 提供商
3. **可测试**: 用 Mock 客户端进行单元测试

## 总结

### 核心概念

1. **Message**: 消息 = 角色 + 内容
2. **Role**: 三种角色 - SYSTEM, USER, ASSISTANT
3. **LLMClient**: 协议，定义统一接口
4. **GLMClient**: 具体实现，封装智谱 AI API

### 关键理解

- LLM 交互本质是**消息交换**
- LLM 是**无状态**的，需要发送完整历史
- **分层设计**让代码更灵活、可测试

### 下一步

- 运行 experiments 中的示例代码
- 尝试添加更多 LLM 适配器
- 阅读源码中的详细注释